In [1]:
import pandas as pd

# 1. Cargar los datos
df = pd.read_csv('../data/la_liga_2014_2025_all_matches_final.csv')

# 2. Imprimir información básica
print("Filas y columnas:", df.shape)
print("\nTipos de datos:\n", df.dtypes.head(10)) # Solo los primeros 10 para no saturar
print("\nValores nulos por columna:\n", df.isnull().sum().head(10))
print("\nFilas duplicadas:", df.duplicated().sum())

Filas y columnas: (4180, 19)

Tipos de datos:
 match_id        int64
date              str
home_team         str
away_team         str
home_goals      int64
away_goals      int64
home_xg       float64
away_xg       float64
home_shots      int64
away_shots      int64
dtype: object

Valores nulos por columna:
 match_id      0
date          0
home_team     0
away_team     0
home_goals    0
away_goals    0
home_xg       0
away_xg       0
home_shots    0
away_shots    0
dtype: int64

Filas duplicadas: 0


In [2]:
# 1. Confirma que SÍ son equipos de La Liga (Barça, Madrid, Atlético...) y no otra cosa
print(df['home_team'].unique())

# 2. Rango de fechas y temporadas cubiertas
df['date'] = pd.to_datetime(df['date'])
print("Rango:", df['date'].min(), "-", df['date'].max())
print("\nPartidos por temporada (año):\n", df['date'].dt.year.value_counts().sort_index())

# 3. Ver las 9 columnas restantes que no salieron en el head() anterior
print("\nTodas las columnas:\n", df.columns.tolist())
print("\nNulos en TODAS las columnas:\n", df.isnull().sum())

<StringArray>
[             'Malaga',             'Sevilla',             'Granada',
             'Almeria',               'Eibar',           'Barcelona',
          'Celta Vigo',             'Levante',         'Real Madrid',
      'Rayo Vallecano',              'Getafe',            'Valencia',
             'Cordoba',       'Athletic Club',     'Atletico Madrid',
            'Espanyol',          'Villarreal', 'Deportivo La Coruna',
       'Real Sociedad',               'Elche',      'Sporting Gijon',
          'Real Betis',          'Las Palmas',             'Osasuna',
             'Leganes',              'Alaves',              'Girona',
     'Real Valladolid',           'SD Huesca',            'Mallorca',
               'Cadiz']
Length: 31, dtype: str
Rango: 2014-08-23 18:00:00 - 2025-05-25 19:00:00

Partidos por temporada (año):
 date
2014    159
2015    390
2016    370
2017    390
2018    380
2019    391
2020    355
2021    407
2022    348
2023    410
2024    381
2025    199
Name: cou

In [6]:
# 1. Ver las 19 columnas completas (para saber con qué features realmente cuentas)
print("Todas las columnas:\n", df.columns.tolist())

# 2. Sanity check de rangos (goles y xG no deberían tener valores absurdos)
print("\nResumen estadístico:\n", df[['home_goals','away_goals','home_xg','away_xg']].describe())

# 3. Balance de clases del resultado — esto es CRÍTICO para el modelo
def resultado(row):
    if row['home_goals'] > row['away_goals']:
        return 'Victoria local'
    elif row['home_goals'] < row['away_goals']:
        return 'Victoria visitante'
    else:
        return 'Empate'

df['resultado'] = df.apply(resultado, axis=1)
print("\nDistribución de resultados:\n", df['resultado'].value_counts(normalize=True))

Todas las columnas:
 ['match_id', 'date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'home_shots', 'away_shots', 'home_sot', 'away_sot', 'home_deep', 'away_deep', 'home_ppda', 'away_ppda', 'home_xpts', 'away_xpts', 'season_mapped', 'resultado']

Resumen estadístico:
         home_goals   away_goals      home_xg      away_xg
count  4180.000000  4180.000000  4180.000000  4180.000000
mean      1.494258     1.132057     1.544172     1.147610
std       1.297453     1.120957     0.923703     0.784594
min       0.000000     0.000000     0.050000     0.000000
25%       1.000000     0.000000     0.860000     0.560000
50%       1.000000     1.000000     1.380000     0.980000
75%       2.000000     2.000000     2.040000     1.560000
max      10.000000     8.000000     6.610000     6.190000

Distribución de resultados:
 resultado
Victoria local        0.453828
Victoria visitante    0.286364
Empate                0.259809
Name: proportion, dtype: float64


In [7]:
home = df[['date','season_mapped','home_team','home_goals','away_goals','home_xg','away_xg']].copy()
home.columns = ['date','season','team','goals_for','goals_against','xg_for','xg_against']
home['is_home'] = 1

away = df[['date','season_mapped','away_team','away_goals','home_goals','away_xg','home_xg']].copy()
away.columns = ['date','season','team','goals_for','goals_against','xg_for','xg_against']
away['is_home'] = 0

team_matches = pd.concat([home, away]).sort_values(['team','date']).reset_index(drop=True)
team_matches['puntos'] = team_matches.apply(
    lambda r: 3 if r['goals_for'] > r['goals_against'] else (1 if r['goals_for'] == r['goals_against'] else 0), axis=1
)

In [8]:
team_matches['forma_goles_favor'] = (
    team_matches.groupby('team')['goals_for']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_matches['forma_goles_contra'] = (
    team_matches.groupby('team')['goals_against']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_matches['forma_xg'] = (
    team_matches.groupby('team')['xg_for']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
team_matches['forma_puntos'] = (
    team_matches.groupby('team')['puntos']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [9]:
forma_cols = ['forma_goles_favor','forma_goles_contra','forma_xg','forma_puntos']

home_forma = team_matches[team_matches['is_home']==1][['date','team'] + forma_cols].copy()
home_forma.columns = ['date','home_team'] + ['home_' + c for c in forma_cols]

away_forma = team_matches[team_matches['is_home']==0][['date','team'] + forma_cols].copy()
away_forma.columns = ['date','away_team'] + ['away_' + c for c in forma_cols]
df_final = df.merge(home_forma, on=['date','home_team'], how='left')
df_final = df_final.merge(away_forma, on=['date','away_team'], how='left')

In [10]:
print("Filas y columnas de df_final:", df_final.shape)
print("\nColumnas de df_final:\n", df_final.columns.tolist())
df_final.head()

Filas y columnas de df_final: (4180, 28)

Columnas de df_final:
 ['match_id', 'date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'home_shots', 'away_shots', 'home_sot', 'away_sot', 'home_deep', 'away_deep', 'home_ppda', 'away_ppda', 'home_xpts', 'away_xpts', 'season_mapped', 'resultado', 'home_forma_goles_favor', 'home_forma_goles_contra', 'home_forma_xg', 'home_forma_puntos', 'away_forma_goles_favor', 'away_forma_goles_contra', 'away_forma_xg', 'away_forma_puntos']


,match_id,date,home_team,away_team,home_goals,away_goals,home_xg,away_xg,home_shots,away_shots,...,season_mapped,resultado,home_forma_goles_favor,home_forma_goles_contra,home_forma_xg,home_forma_puntos,away_forma_goles_favor,away_forma_goles_contra,away_forma_xg,away_forma_puntos
0,5826,2014-08-23 18:00:00,Malaga,Athletic Club,1,0,1.32,1.14,12,12,...,2014-15,Victoria local,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5827,2014-08-23 20:00:00,Sevilla,Valencia,1,1,1.17,1.75,12,11,...,2014-15,Empate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5828,2014-08-23 20:00:00,Granada,Deportivo La Coruna,2,1,0.55,0.38,10,8,...,2014-15,Victoria local,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5829,2014-08-23 22:00:00,Almeria,Espanyol,1,1,0.98,0.40,19,12,...,2014-15,Empate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5830,2014-08-24 18:00:00,Eibar,Real Sociedad,1,0,0.47,0.98,12,19,...,2014-15,Victoria local,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
team_matches[team_matches['team']=='Barcelona'].head(10)

,date,season,team,goals_for,goals_against,xg_for,xg_against,is_home,puntos,forma_goles_favor,forma_goles_contra,forma_xg,forma_puntos
1254,2014-08-24 20:00:00,2014-15,Barcelona,3,0,1.54,0.11,1,3,NaN,NaN,NaN,NaN
1255,2014-08-31 18:00:00,2014-15,Barcelona,1,0,3.13,1.11,0,3,3.00,0.0,1.540000,3.0
1256,2014-09-13 15:00:00,2014-15,Barcelona,2,0,2.18,0.10,1,3,2.00,0.0,2.335000,3.0
1257,2014-09-21 20:00:00,2014-15,Barcelona,5,0,3.82,0.44,0,3,2.00,0.0,2.283333,3.0
1258,2014-09-24 21:00:00,2014-15,Barcelona,0,0,0.65,0.28,0,1,2.75,0.0,2.667500,3.0
1259,2014-09-27 17:00:00,2014-15,Barcelona,6,0,4.93,0.45,1,3,2.20,0.0,2.264000,2.6
1260,2014-10-04 17:00:00,2014-15,Barcelona,2,0,2.98,0.52,0,3,2.80,0.0,2.942000,2.6
1261,2014-10-18 19:00:00,2014-15,Barcelona,3,0,3.09,0.93,1,3,3.00,0.0,2.912000,2.6
1262,2014-10-25 17:00:00,2014-15,Barcelona,1,3,1.30,1.88,0,0,3.20,0.0,3.094000,2.6
1263,2014-11-01 19:00:00,2014-15,Barcelona,0,1,2.66,1.56,1,0,2.40,0.6,2.590000,2.0


In [12]:
print("NaN en df_final:\n", df_final[['home_forma_goles_favor','away_forma_goles_favor']].isnull().sum())
print("\nTotal de filas:", df_final.shape[0])
print("\nEjemplo de filas con NaN:\n", df_final[df_final['home_forma_goles_favor'].isnull()][['date','home_team','away_team']].head(10))

NaN en df_final:
 home_forma_goles_favor    15
away_forma_goles_favor    16
dtype: int64

Total de filas: 4180

Ejemplo de filas con NaN:
                  date       home_team            away_team
0 2014-08-23 18:00:00          Malaga        Athletic Club
1 2014-08-23 20:00:00         Sevilla             Valencia
2 2014-08-23 20:00:00         Granada  Deportivo La Coruna
3 2014-08-23 22:00:00         Almeria             Espanyol
4 2014-08-24 18:00:00           Eibar        Real Sociedad
5 2014-08-24 20:00:00       Barcelona                Elche
6 2014-08-24 20:00:00      Celta Vigo               Getafe
7 2014-08-24 22:00:00         Levante           Villarreal
8 2014-08-25 19:00:00     Real Madrid              Cordoba
9 2014-08-25 21:00:00  Rayo Vallecano      Atletico Madrid


In [13]:
# Encuentra la primera aparición de cada equipo en tu dataset (ya está ordenado por fecha)
primera_aparicion = team_matches.groupby('team').first()

fecha_inicio = team_matches['date'].min()

# Equipos que debutan justo en la jornada 1 (2014-15)
debut_jornada1 = primera_aparicion[primera_aparicion['date'] <= fecha_inicio + pd.Timedelta(days=3)]
print("Equipos que debutan en la jornada 1:", len(debut_jornada1))

# Equipos que aparecen por primera vez DESPUÉS de la jornada 1 (posibles ascensos "nuevos" para el dataset)
debut_tardio = primera_aparicion[primera_aparicion['date'] > fecha_inicio + pd.Timedelta(days=3)]
print("\nEquipos con primera aparición posterior (candidatos a 'Córdoba'):\n", debut_tardio[['date','is_home']])

Equipos que debutan en la jornada 1: 20

Equipos con primera aparición posterior (candidatos a 'Córdoba'):
                                date  is_home
team                                        
Alaves          2016-08-22 00:15:00        0
Cadiz           2020-09-12 19:00:00        1
Girona          2017-08-19 19:15:00        1
Las Palmas      2015-08-22 22:30:00        0
Leganes         2016-08-22 22:00:00        0
Mallorca        2019-08-17 19:00:00        1
Osasuna         2016-08-19 22:45:00        0
Real Betis      2015-08-24 00:30:00        1
Real Valladolid 2018-08-17 21:15:00        0
SD Huesca       2018-08-19 19:15:00        0
Sporting Gijon  2015-08-23 22:30:00        1
